# 02 — Mesh Inspection
Check mesh quality after marching cubes extraction and registration.

In [ ]:
import sys; sys.path.insert(0, '..')
import trimesh
import numpy as np
from pathlib import Path

mesh_dir = Path('data/meshes')
sample = sorted(mesh_dir.glob('*/ed_lv.ply'))[0]
mesh = trimesh.load(str(sample), process=False)
print(f'Vertices: {len(mesh.vertices)}, Faces: {len(mesh.faces)}')
print(f'Is watertight: {mesh.is_watertight}')
print(f'Bounds: {mesh.bounds}')

In [ ]:
# Vertex count distribution
import matplotlib.pyplot as plt

counts = [len(trimesh.load(str(p), process=False).vertices) for p in mesh_dir.glob('*/ed_lv.ply')]
plt.hist(counts, bins=20)
plt.xlabel('Vertex count')
plt.ylabel('Frequency')
plt.title('LV mesh vertex count distribution (ED)')
plt.axvline(np.median(counts), color='red', linestyle='--', label=f'Median: {np.median(counts):.0f}')
plt.legend()
plt.tight_layout()

## Mesh Visualization

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
import numpy as np

def plot_mesh(mesh, title='Mesh', ax=None, alpha=0.3, color='steelblue'):
    if ax is None:
        fig = plt.figure(figsize=(7, 7))
        ax = fig.add_subplot(111, projection='3d')
    tris = mesh.vertices[mesh.faces]
    poly = Poly3DCollection(tris, alpha=alpha, facecolor=color, edgecolor='none')
    ax.add_collection3d(poly)
    verts = mesh.vertices
    ax.set_xlim(verts[:, 0].min(), verts[:, 0].max())
    ax.set_ylim(verts[:, 1].min(), verts[:, 1].max())
    ax.set_zlim(verts[:, 2].min(), verts[:, 2].max())
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
    ax.set_title(title)
    return ax

ax = plot_mesh(mesh, title=f'ED LV — {sample.parent.name}\n{len(mesh.vertices)} verts, {len(mesh.faces)} faces')
plt.tight_layout()
plt.show()

In [ ]:
# Compare ED vs ES for the same patient
patient_dir = sample.parent
ed_mesh = trimesh.load(str(patient_dir / 'ed_lv.ply'), process=False)
es_mesh = trimesh.load(str(patient_dir / 'es_lv.ply'), process=False)

fig = plt.figure(figsize=(14, 7))
ax1 = fig.add_subplot(121, projection='3d')
ax2 = fig.add_subplot(122, projection='3d')

plot_mesh(ed_mesh, title=f'ED (end-diastole)\n{len(ed_mesh.vertices)} verts', ax=ax1, color='steelblue')
plot_mesh(es_mesh, title=f'ES (end-systole)\n{len(es_mesh.vertices)} verts', ax=ax2, color='tomato')

fig.suptitle(f'LV — {patient_dir.name}', fontsize=13)
plt.tight_layout()
plt.show()

print(f'Volume ED: {ed_mesh.volume:.1f} mm³  |  ES: {es_mesh.volume:.1f} mm³')
print(f'Ejection fraction ≈ {(ed_mesh.volume - es_mesh.volume) / ed_mesh.volume * 100:.1f}%')

In [ ]:
fig_dir = Path('../report/figures')
fig_dir.mkdir(parents=True, exist_ok=True)

# --- Single mesh ---
fig1, ax1 = plt.subplots(1, 1, figsize=(5, 5), subplot_kw={'projection': '3d'})
plot_mesh(mesh, title=f'ED LV — {sample.parent.name}\n{len(mesh.vertices)} verts, {len(mesh.faces)} faces', ax=ax1)
fig1.tight_layout()
fig1.savefig(fig_dir / 'ed_lv_single.pdf', bbox_inches='tight')
fig1.savefig(fig_dir / 'ed_lv_single.png', dpi=150, bbox_inches='tight')
plt.close(fig1)

# --- ED vs ES comparison ---
fig2, (axA, axB) = plt.subplots(1, 2, figsize=(10, 5), subplot_kw={'projection': '3d'})
plot_mesh(ed_mesh, title=f'ED (end-diastole)\n{len(ed_mesh.vertices)} verts', ax=axA, color='steelblue')
plot_mesh(es_mesh, title=f'ES (end-systole)\n{len(es_mesh.vertices)} verts', ax=axB, color='tomato')
fig2.suptitle(f'LV — {patient_dir.name}', fontsize=13)
fig2.tight_layout()
fig2.savefig(fig_dir / 'ed_es_comparison.pdf', bbox_inches='tight')
fig2.savefig(fig_dir / 'ed_es_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig2)

print(f'Saved figures to {fig_dir.resolve()}')